# Calibrate using TensorFlow

In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd

import fates_calibration_library.emulator_functions as em
import fates_calibration_library.utils as utils
from fates_calibration_library.TFClass import TFEmulator

import importlib

2025-07-02 10:37:48.099172: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-02 10:37:48.100848: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-02 10:37:48.127002: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-02 10:37:48.127733: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-02 10:37:50.695969: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT

## Set Up
Load files, set up ensemble information

In [2]:
# directories
mesh_dir = '/glade/work/afoster/FATES_calibration/mesh_files'
emulator_dir = '/glade/work/afoster/FATES_calibration/emulators'
fig_dir = '/glade/work/afoster/FATES_calibration/figures'
param_dir = '/glade/work/afoster/FATES_calibration/parameter_files'

# default parameter file
default_param = xr.open_dataset(os.path.join(param_dir, 
                                             'fates_params_default_sci.1.85.1_api.40.0.0_crops.nc'))
all_pfts = [str(pft).replace("b'", "").replace("'", "").strip() for pft in default_param.fates_pftname.values]

# normalized values for parameters
default_norm = pd.read_csv(os.path.join(param_dir, 'normalized_parameters.csv'), index_col=[0])

# variables to calibrate
calibration_vars = ['GPP', 'EFLX_LH_TOT', 'FSH', 'EF']

# information about variables
obs_config_file = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/ilamb_conversion.yaml'
obs_config = utils.get_config_file(obs_config_file)

# PFT ids
pft_id_config = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/fates_pft_ids.yaml'
pft_ids = utils.get_config_file(pft_id_config)

In [3]:
# information about each ensemble
ens_dict = {'dompft':
            {'mesh_file': os.path.join(mesh_dir, 'dominant_grid_mesh.nc'),
             'land_mask_file': os.path.join(mesh_dir, 'dominant_grid.nc'),
             'lhc_key_file': os.path.join(param_dir, 'fates_lh', 'fates_lh_key.csv'),
             'pfts': [1, 2, 3, 12, 13, 14],
             'obs_df': os.path.join(mesh_dir, 'dominant_grid.csv'),
            }
           }

In [4]:
# choose ensemble
ensemble = 'dompft'

### Load Latin Hypercube Key

In [5]:
lhc_key = pd.read_csv(ens_dict[ensemble]['lhc_key_file'], index_col=[0])
lhc_key = lhc_key.drop(columns=['ensemble'])
param_names = lhc_key.columns
num_params = len(param_names)

### Load Observations

In [6]:
obs = pd.read_csv(ens_dict[ensemble]['obs_df'], index_col=[0])

### Load Parameter Sensitivity

In [7]:
sens_df = pd.read_csv(os.path.join(emulator_dir, f'sensitivity_df_{ensemble}.csv'), index_col=[0])

## Calibration

In [113]:
pft = 1
pft_name = all_pfts[pft-1]
pft_id = pft_ids[pft_name]

In [116]:
# get observations for this pft
obs_pft = obs[obs.pft == pft_name]

# get sensitivity data for this pft
sens_pft = sens_df[sens_df.pft == pft_id]

# get default values for this pft
default_pft = default_norm[default_norm.pft == pft]
default_pft = default_pft.drop(columns=['pft'])
X_default_all = default_pft.to_numpy().flatten()

In [117]:
# stack targets, sds, and emulators for all variables
targets = []
sds = []
emulators = []
for variable in calibration_vars:
    
    # observations for this pft and variable
    obs_mean, obs_sd = em.get_obs_mean_and_sd(obs_pft, obs_config[variable]['var'])
    
    # convert to tf objects
    targets.append(obs_mean)
    sds.append(obs_sd)

    # load the emulator
    emulators.append(TFEmulator(emulator_dir, pft=pft_id, variable=variable))

In [118]:
config = {
    'maxiter': 5000,
    'epsilon': 0.5,
    'lambda_penalty': None,
    'barrier_strength': 0,
    'loss_fn': em.implausibility_loss,
    'default_penalty_fn': em.default_penalty_l1,
    'barrier_penalty_fn': em.barrier_penalty,
    'tol': 1e-3
}

In [124]:
importlib.reload(em)

<module 'fates_calibration_library.emulator_functions' from '/glade/work/afoster/FATES_calibration/fates_calibration_library/fates_calibration_library/emulator_functions.py'>

In [ ]:
all_results = run_batch_optimization(emulators, targets, sds, fixed_indices, X_default_all, num_optimize, config, num_batch=10000)

In [149]:
df_wide = all_results.pivot(index='batch', columns='parameter', values='values')